In [ ]:
# 1) load everything
import pandas as pd

sheets = pd.read_excel("../data/online_retail_II.xlsx", sheet_name=None) # reads both sheets, ../data -> go up one folder
df = pd.concat(sheets.values(), ignore_index=True) # stacks both sheets into one table renumbering from 0

df.columns = ["invoice", "stock_code", "description", "quantity", "invoice_date", "price", "customer_id", "country"] # renames columns to avoid awkwardness in SQL
df.to_pickle("../data/raw.pkl") # saves fast-loading copy
df.shape

(1067371, 8)

In [ ]:
# 2a) first look
df.head(10) # shows first 10 rows

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [ ]:
# 2b) first look 
df.info() # lists each column/data type/number of values

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  object        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[us]
 5   price         1067371 non-null  float64       
 6   customer_id   824364 non-null   float64       
 7   country       1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [ ]:
# 3) data quality table
# each line asks yes/no for each row and counts the yeses
checks = { 
    "Total rows": len(df),
    "Exact duplicate rows": df.duplicated().sum(),
    "Missing customer ID": df["customer_id"].isna().sum(), # isna flags empty cells
    "Missing description": df["description"].isna().sum(),
    "Cancellation lines (invoice starts with C)": df["invoice"].astype(str).str.startswith("C").sum(), # turns invoice num into text
    "Zero or negative quantity": (df["quantity"] <= 0).sum(),
    "Zero or negative price": (df["price"] <= 0).sum(),
}

report = pd.Series(checks, name="rows").to_frame() # turns into a table
report["pct_of_total"] = (report["rows"] / len(df) * 100).round(2) # adds percentage column
report.to_csv("../reports/data_quality_summary.csv", index_label="check") # saves table to reports folder
report

,rows,pct_of_total
Total rows,1067371,100.00
Exact duplicate rows,34335,3.22
Missing customer ID,243007,22.77
Missing description,4382,0.41
Cancellation lines (invoice starts with C),19494,1.83
Zero or negative quantity,22950,2.15
Zero or negative price,6207,0.58


In [6]:
# 4) scope of the data
print("Date range:", df["invoice_date"].min(), "to", df["invoice_date"].max())
print("Customers:", df["customer_id"].nunique()) # nunique() counts distinct values
print("Countries:", df["country"].nunique())
print("Products:", df["stock_code"].nunique())

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Customers: 5942
Countries: 43
Products: 5305
